In [1]:
# @title Install required packages
!pip install -U qwen-tts soundfile gradio -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 1.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.3/73.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 9.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the package

In [2]:
# @title Import libraries
import torch
import soundfile as sf
import numpy as np
import gradio as gr
from qwen_tts import Qwen3TTSModel


    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    



********
********
 


In [3]:
# @title Load Qwen3‑TTS model (once loaded no need to run again)
print("Loading Qwen3‑TTS")
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-1.7B-Base",
    device_map="cuda:0",
    dtype=torch.float16,
)
print("Model loaded")

Loading Qwen3‑TTS


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/245 [00:00<?, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/127 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

Model loaded


In [4]:
# @title Voice normalisation for more Human Like
def clone_voice(ref_audio_path, ref_text, target_text):
    """
    ref_audio_path : str – path to reference WAV file
    ref_text       : str – exact transcript of the reference audio
    target_text    : str – text to synthesise with the cloned voice
    Returns        : tuple (sample_rate, audio_array) for Gradio
    """
    wavs, sr = model.generate_voice_clone(
        text=target_text,
        language="English",
        ref_audio=ref_audio_path,
        ref_text=ref_text,
    )
    audio = wavs[0]

    target_dBFS = -1.0
    target_amp = 10 ** (target_dBFS / 20.0)
    peak = np.max(np.abs(audio))
    if peak > 0:
        audio = audio * (target_amp / peak)

    return sr, audio

In [5]:
# @title Gradio UI
with gr.Blocks(title="Voice Cloning with Qwen3‑TTS") as demo:

    gr.HTML("""
    <div style="display: flex; align-items: center; gap: 10px; margin-bottom: 10px;">
        <a href="https://github.com/HarishDevLab/HarishDevLab/blob/main/cd.txt" target="_blank">
            <img src="https://github.githubassets.com/assets/apple-touch-icon-144x144-b882e354c005.png"
                 alt="GitHub" width="28" style="vertical-align: middle;">
        </a>
        <a href="https://github.com/HarishDevLab/HarishDevLab/blob/main/cd.txt" target="_blank"
           style="text-decoration: none; color: inherit;">
            <span style="font-size: 1.4em; font-weight: bold;">Voice Cloning with Qwen3‑TTS</span>
        </a>
    </div>
    """)

    with gr.Row():
        with gr.Column():
            ref_audio = gr.Audio(label="Reference Audio", type="filepath")
            ref_text = gr.Textbox(
                label="Reference Text (the text spoken in the reference audio)",
                placeholder="Reference Text",
                lines=3,
            )
            target_text = gr.Textbox(
                label="Text to Synthesise",
                placeholder="Text to Synthesise",
                lines=5,
            )
            btn = gr.Button("Generate Voice", variant="primary")
        with gr.Column():
            output_audio = gr.Audio(label="Cloned Voice", type="numpy", interactive=False)

    btn.click(
        fn=clone_voice,
        inputs=[ref_audio, ref_text, target_text],
        outputs=output_audio,
    )

In [6]:
# @title Launch Gradio
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://817b3f5fc6e04f125b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.
/usr/local/lib/python3.12/dist-packages/gradio/processing_utils.py:724: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://817b3f5fc6e04f125b.gradio.live
